# Notas 1: Métricas entre genes.
Este _notebook_ presenta un conjunto de métricas útiles para estimar relaciones entre genes desde distintas fuentes de evidencia biológica y molecular. El objetivo es disponer de descripciones conceptuales, fórmulas y criterios de interpretación que sirvan como base para implementar matrices de similitud o distancia entre pares de genes. Asimismo, la integración
de código fuente para implementar el cálculo de estas métricas.

Las métricas consideradas son: interacción proteína–proteína en STRING, puntaje de asociación gen–enfermedad de DISGENET, co-pertenencia a rutas de KEGG, similitud de secuencia mediante BLAST y métricas de gene ontology. En un flujo de clustering multiobjetivo, estas métricas pueden utilizarse como fuentes complementarias para representar coherencia biológica, relaciones funcionales o proximidad molecular entre genes.

## Métrica 1: Protein-Protein Interaction (STRING)
La métrica de interacción proteína–proteína de STRING representa el nivel de confianza de que dos proteínas se encuentren asociadas funcional o físicamente. En STRING, una asociación no necesariamente implica contacto físico directo: también puede reflejar participación en un mismo proceso biológico, ruta, complejo funcional o relación inferida por evidencia computacional y experimental. STRING integra distintos canales de evidencia, entre ellos experimentos, bases de datos curadas, coexpresión, minería de texto, vecindad génica, fusión génica, coocurrencia filogenética y evidencia transferida por ortología. Cada canal entrega un puntaje probabilístico y STRING los combina en un puntaje final denominado `combined_score`. Este valor suele entregarse escalado entre 150 y 999 en los archivos descargables, aunque conceptualmente corresponde a una confianza aproximada entre 0.150 y 0.999 (Szklarczyk et al. , 2025) (STRING Help, s. f.).

Una interpretación práctica habitual es considerar umbrales de confianza: valores cercanos a 0.4 representan confianza media, mientras que valores desde 0.7 suelen interpretarse como confianza alta. Estos umbrales no deben tratarse como reglas universales, ya que dependen del organismo, de la densidad de la red y del objetivo del análisis (STRING Help, s. f.).

La combinación de los canales de evidencia considera una probabilidad previa de asociación aleatoria, indicada por STRING como (STRING Help, s. f.):
$$p = 0.041$$

Para cada canal de evidencia se elimina primero la contribución de esta probabilidad previa:

$$s'_c = \frac{s_c - p}{1-p}$$

Luego, los canales ajustados se combinan bajo el supuesto de independencia aproximada:

$$S' = 1 - \prod_{c}(1-s'_c)$$

Finalmente, se reincorpora una vez la probabilidad previa:

$$S = S' + p(1-S')$$

donde $s_c$ es el puntaje del canal $c$, $s'_c$ es el puntaje ajustado, $S'$ es el puntaje combinado ajustado y $S$ corresponde al puntaje combinado final.

En análisis a nivel de genes, este puntaje se usa normalmente después de mapear cada gen a su producto proteico correspondiente. En muchos casos se asume una relación canónica uno a uno entre gen y proteína; sin embargo, se debe tener cuidado con genes que codifican múltiples isoformas, genes sin mapeo claro o proteínas fusionadas. Para construir una matriz gen–gen, una opción simple es usar directamente el `combined_score` normalizado a $[0,1]$. Si se requiere una matriz de distancia, puede transformarse como (STRING Help, s. f.):

$$d(g_i,g_j)=1-S(g_i,g_j)$$

donde valores cercanos a 0 indican alta asociación funcional o física entre los genes/proteínas comparados.

In [8]:
import time
from io import StringIO
from typing import List, Tuple

import numpy as np
import pandas as pd
import requests


STRING_API_URL = "https://version-12-0.string-db.org/api"
OUTPUT_FORMAT = "tsv"


def map_genes_to_string_ids(
    genes: List[str],
    species: int = 9606,
    caller_identity: str = "gclusters_characterization"
) -> pd.DataFrame:
    """
    Map gene symbols to STRING protein identifiers.

    Parameters
    ----------
    genes : list of str
        Gene symbols or protein identifiers.
    species : int
        NCBI taxonomy identifier. Human = 9606.
    caller_identity : str
        Identifier for your tool/script, recommended by STRING.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns such as queryItem, stringId and preferredName.
    """
    request_url = f"{STRING_API_URL}/{OUTPUT_FORMAT}/get_string_ids"

    params = {
        "identifiers": "\r".join(genes),
        "species": species,
        "limit": 1,
        "echo_query": 1,
        "caller_identity": caller_identity,
    }

    response = requests.post(request_url, data=params, timeout=60)
    response.raise_for_status()

    if not response.text.strip():
        return pd.DataFrame()

    return pd.read_csv(StringIO(response.text), sep="\t")


def get_string_network(
    string_ids: List[str],
    species: int = 9606,
    required_score: int = 400,
    network_type: str = "functional",
    caller_identity: str = "gclusters_characterization"
) -> pd.DataFrame:
    """
    Retrieve STRING network interactions for a list of STRING IDs.

    Parameters
    ----------
    string_ids : list of str
        STRING protein identifiers.
    species : int
        NCBI taxonomy identifier. Human = 9606.
    required_score : int
        Minimum confidence score in STRING scale 0-1000.
        Common values: 150 low, 400 medium, 700 high, 900 highest.
    network_type : str
        "functional" for the full STRING association network or
        "physical" for physical subnetworks.
    caller_identity : str
        Identifier for your tool/script.

    Returns
    -------
    pandas.DataFrame
        Interaction table returned by STRING.
    """
    request_url = f"{STRING_API_URL}/{OUTPUT_FORMAT}/network"

    params = {
        "identifiers": "\r".join(string_ids),
        "species": species,
        "required_score": required_score,
        "network_type": network_type,
        "caller_identity": caller_identity,
    }

    response = requests.post(request_url, data=params, timeout=60)
    response.raise_for_status()

    if not response.text.strip():
        return pd.DataFrame()

    return pd.read_csv(StringIO(response.text), sep="\t")


def build_string_ppi_similarity_matrix(
    genes: List[str],
    species: int = 9606,
    required_score: int = 400,
    network_type: str = "functional",
    missing_value: float = 0.0,
    caller_identity: str = "gclusters_characterization"
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build a gene-gene similarity matrix from STRING PPI confidence scores.

    Parameters
    ----------
    genes : list of str
        Input gene symbols.
    species : int
        NCBI taxonomy identifier.
    required_score : int
        Minimum STRING confidence score in 0-1000 scale.
    network_type : str
        "functional" or "physical".
    missing_value : float
        Value assigned to pairs without STRING interaction.
    caller_identity : str
        Identifier for the script.

    Returns
    -------
    similarity_matrix : pandas.DataFrame
        Symmetric gene-gene similarity matrix in range [0, 1].
    mapping_df : pandas.DataFrame
        Mapping between input genes and STRING identifiers.
    network_df : pandas.DataFrame
        Raw STRING network table.
    """
    genes = list(dict.fromkeys(genes))  # remove duplicates, preserve order

    mapping_df = map_genes_to_string_ids(
        genes=genes,
        species=species,
        caller_identity=caller_identity,
    )

    if mapping_df.empty:
        raise ValueError("No genes could be mapped to STRING identifiers.")

    # Keep one STRING ID per input gene
    mapping_df = mapping_df.drop_duplicates(subset=["queryItem"], keep="first")

    query_to_string = dict(zip(mapping_df["queryItem"], mapping_df["stringId"]))
    string_to_query = dict(zip(mapping_df["stringId"], mapping_df["queryItem"]))

    mapped_genes = [g for g in genes if g in query_to_string]
    string_ids = [query_to_string[g] for g in mapped_genes]

    if len(string_ids) < 2:
        raise ValueError("At least two mapped genes are required to build a matrix.")

    time.sleep(1)

    network_df = get_string_network(
        string_ids=string_ids,
        species=species,
        required_score=required_score,
        network_type=network_type,
        caller_identity=caller_identity,
    )

    similarity_matrix = pd.DataFrame(
        missing_value,
        index=mapped_genes,
        columns=mapped_genes,
        dtype=float,
    )

    np.fill_diagonal(similarity_matrix.values, 1.0)

    if network_df.empty:
        return similarity_matrix, mapping_df, network_df

    for _, row in network_df.iterrows():
        protein_a = row["stringId_A"]
        protein_b = row["stringId_B"]

        gene_a = string_to_query.get(protein_a)
        gene_b = string_to_query.get(protein_b)

        if gene_a is None or gene_b is None:
            continue

        # STRING API usually returns score in [0, 1]
        score = float(row["score"])

        similarity_matrix.loc[gene_a, gene_b] = score
        similarity_matrix.loc[gene_b, gene_a] = score

    return similarity_matrix, mapping_df, network_df

In [9]:
genes = [
    "TP53",
    "BRCA1",
    "BRCA2",
    "EGFR",
    "MYC",
    "CDK2",
    "MDM2",
]

sim_matrix, mapping, interactions = build_string_ppi_similarity_matrix(
    genes=genes,
    species=9606,
    required_score=400,
    network_type="functional",
    caller_identity="gclusters_characterization"
)

print(sim_matrix)
print(mapping[["queryItem", "stringId", "preferredName"]])
print(interactions.head())

        TP53  BRCA1  BRCA2   EGFR    MYC   CDK2   MDM2
TP53   1.000  0.999  0.995  0.943  0.997  0.987  0.999
BRCA1  0.999  1.000  0.999  0.800  0.999  0.984  0.794
BRCA2  0.995  0.999  1.000  0.678  0.759  0.946  0.702
EGFR   0.943  0.800  0.678  1.000  0.875  0.566  0.897
MYC    0.997  0.999  0.759  0.875  1.000  0.960  0.813
CDK2   0.987  0.984  0.946  0.566  0.960  1.000  0.849
MDM2   0.999  0.794  0.702  0.897  0.813  0.849  1.000
  queryItem              stringId preferredName
0      TP53  9606.ENSP00000269305          TP53
1     BRCA1  9606.ENSP00000418960         BRCA1
2     BRCA2  9606.ENSP00000369497         BRCA2
3      EGFR  9606.ENSP00000275493          EGFR
4       MYC  9606.ENSP00000478887           MYC
5      CDK2  9606.ENSP00000266970          CDK2
6      MDM2  9606.ENSP00000258149          MDM2
             stringId_A            stringId_B preferredName_A preferredName_B  \
0  9606.ENSP00000258149  9606.ENSP00000369497            MDM2           BRCA2   
1  9606.ENSP00

## Métrica 2: Gene–Disease Association Score (DISGENET)
El _Gene–Disease Association Score_ (GDA Score) de DISGENET cuantifica la fuerza de evidencia que respalda la asociación entre un gen y una enfermedad. A diferencia de una métrica estrictamente molecular entre dos genes, el GDA Score relaciona cada gen con una entidad clínica o fenotipo específico. Por ello, para comparaciones gen–gen debe definirse primero una enfermedad de interés o construir un vector de asociaciones gen–enfermedad para cada gen. DISGENET integra evidencia proveniente de bases de datos curadas, literatura científica, modelos animales, evidencia inferida y ensayos clínicos. El puntaje se expresa en el rango $[0,1]$, donde valores más altos indican asociaciones más respaldadas por evidencia independiente, confiable y repetida. En términos exploratorios, valores cercanos o inferiores a 0.3 suelen representar evidencia limitada o emergente, mientras que valores cercanos a 1 sugieren una asociación fuertemente respaldada (Piñero et al., 2019) (DISGENET, s. f.).

Una forma general de expresar el puntaje es (DISGENET, s. f.):

$$GDA(g,d)=w_C C + w_L L + w_M M + w_I I + w_T T$$

donde $g$ es el gen, $d$ la enfermedad, $C$ representa evidencia de fuentes curadas, $L$ evidencia bibliográfica, $M$ evidencia desde modelos animales, $I$ evidencia inferida y $T$ evidencia de ensayos clínicos o casos clínicos. Los pesos $w$ reflejan la confiabilidad relativa de cada tipo de evidencia.

Para convertir esta información en una métrica entre genes existen dos alternativas frecuentes:

**1. Asociación respecto a una enfermedad objetivo.** Si se estudia una enfermedad específica $d$, cada gen recibe su puntaje $GDA(g,d)$. La similitud entre dos genes puede definirse como la cercanía entre sus niveles de asociación (DISGENET, s. f.):

$$sim_{GDA}(g_i,g_j \mid d)=1-|GDA(g_i,d)-GDA(g_j,d)|$$

Esta formulación compara si ambos genes tienen evidencia similar respecto a la misma enfermedad, aunque no implica que interactúen biológicamente.

**2. Similitud por perfiles de enfermedades.** Si cada gen se representa mediante un vector de puntajes frente a múltiples enfermedades, se puede calcular similitud entre genes usando coseno, correlación de Spearman/Pearson o Jaccard ponderado sobre las enfermedades asociadas (DISGENET, s. f.):

$$sim(g_i,g_j)=\cos(\vec{GDA}_{g_i},\vec{GDA}_{g_j})$$

Esta segunda alternativa es más adecuada cuando el objetivo es detectar genes con perfiles patológicos similares o pleiotropía compartida. En cualquier caso, los resultados deben interpretarse como similitud de evidencia clínica o biomédica, no como interacción física directa.

In [ ]:
#### Se debe hacer a mano.

## Métrica 3 - Pathway Co-membership Score (KEGG)
El _Pathway Co-membership Score_ mide la similitud entre genes a partir de las rutas biológicas en las que participan. La idea central es que dos genes pueden considerarse funcionalmente relacionados si aparecen anotados en una o más rutas comunes de KEGG, ya que dichas rutas representan procesos celulares, metabólicos, regulatorios o de señalización. La forma más directa de calcular esta similitud es mediante el índice de Jaccard entre los conjuntos de rutas asociadas a cada gen (Parikh et al., 2012b):

$$score(g_i,g_j)=\frac{|Path(g_i)\cap Path(g_j)|}{|Path(g_i)\cup Path(g_j)|}$$

donde $Path(g_i)$ y $Path(g_j)$ corresponden a los conjuntos de rutas KEGG en las que participan los genes $g_i$ y $g_j$. El valor resultante se encuentra en $[0,1]$: un valor 0 indica que no comparten rutas anotadas y un valor 1 indica que ambos genes participan exactamente en el mismo conjunto de rutas (Parikh et al., 2012b).

Es importante corregir la fórmula original: la similitud de Jaccard se calcula como intersección dividida por unión, no al revés. Usar $|A\cup B|/|A\cap B|$ produciría valores mayores que 1 y no sería una medida de similitud normalizada.

Un umbral de 0.5 puede utilizarse como criterio exigente para considerar una relación funcional relativamente fuerte, ya que implica que al menos la mitad de las rutas combinadas son compartidas. Sin embargo, este umbral debe justificarse empíricamente, porque genes altamente específicos pueden compartir solo una ruta relevante y aun así ser funcionalmente importantes.

Para construir una matriz de distancia biológica a partir de esta métrica puede utilizarse:

$$d_{KEGG}(g_i,g_j)=1-score(g_i,g_j)$$

Esta métrica es interpretable y simple, pero depende fuertemente de la cobertura de KEGG. Genes poco estudiados o con baja anotación pueden aparecer artificialmente alejados de otros genes, por falta de información más que por ausencia real de relación biológica.

In [1]:
from __future__ import annotations

import time
from typing import Dict, Iterable, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
import requests


KEGG_REST_URL = "https://rest.kegg.jp"


def _request_kegg_text(endpoint: str, sleep_seconds: float = 0.5) -> str:
    """
    Request a KEGG REST endpoint and return plain text.

    Parameters
    ----------
    endpoint : str
        KEGG endpoint beginning with '/'.
    sleep_seconds : float
        Delay after each request.

    Returns
    -------
    str
        Plain-text response from KEGG.
    """
    url = KEGG_REST_URL + endpoint

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    time.sleep(sleep_seconds)

    return response.text


def _parse_kegg_link_response(text: str) -> List[Tuple[str, str]]:
    """
    Parse a KEGG tab-delimited link response.

    Parameters
    ----------
    text : str
        Text returned by KEGG REST.

    Returns
    -------
    list of tuple
        Pairs from the two returned columns.
    """
    pairs = []

    for line in text.strip().splitlines():
        parts = line.strip().split("\t")

        if len(parts) == 2:
            pairs.append((parts[0], parts[1]))

    return pairs

def get_kegg_gene_pathways(
    organism: str = "hsa",
    sleep_seconds: float = 0.5,
) -> Dict[str, Set[str]]:
    """
    Retrieve KEGG pathways associated with each gene for an organism.

    Parameters
    ----------
    organism : str
        KEGG organism code. Human = 'hsa'.
    sleep_seconds : float
        Delay after request.

    Returns
    -------
    dict
        Dictionary mapping KEGG gene IDs to sets of pathway IDs.
        Example:
        {
            'hsa:7157': {'path:hsa04115', 'path:hsa05200'},
            ...
        }
    """
    text = _request_kegg_text(
        endpoint=f"/link/pathway/{organism}",
        sleep_seconds=sleep_seconds,
    )

    pairs = _parse_kegg_link_response(text)

    gene_to_pathways: Dict[str, Set[str]] = {}

    for gene_id, pathway_id in pairs:
        gene_to_pathways.setdefault(gene_id, set()).add(pathway_id)

    return gene_to_pathways

def pathway_comembership_score(
    pathways_a: Set[str],
    pathways_b: Set[str],
) -> float:
    """
    Compute pathway co-membership score using Jaccard similarity.

    Parameters
    ----------
    pathways_a : set of str
        Pathways associated with gene A.
    pathways_b : set of str
        Pathways associated with gene B.

    Returns
    -------
    float
        Jaccard similarity between pathway sets.
    """
    union = pathways_a | pathways_b

    if not union:
        return 0.0

    intersection = pathways_a & pathways_b

    return len(intersection) / len(union)

def build_kegg_pathway_comembership_matrix_from_kegg_ids(
    kegg_gene_ids: List[str],
    organism: str = "hsa",
    sleep_seconds: float = 0.5,
) -> Tuple[pd.DataFrame, Dict[str, Set[str]]]:
    """
    Build a gene-gene Pathway Co-membership Score matrix from KEGG gene IDs.

    Parameters
    ----------
    kegg_gene_ids : list of str
        KEGG gene IDs, e.g. ['hsa:7157', 'hsa:672'].
    organism : str
        KEGG organism code.
    sleep_seconds : float
        Delay after KEGG requests.

    Returns
    -------
    similarity_matrix : pandas.DataFrame
        Gene-gene similarity matrix.
    selected_gene_pathways : dict
        Mapping from KEGG gene ID to pathway set.
    """
    kegg_gene_ids = list(dict.fromkeys(str(g).strip() for g in kegg_gene_ids if str(g).strip()))

    gene_to_pathways = get_kegg_gene_pathways(
        organism=organism,
        sleep_seconds=sleep_seconds,
    )

    selected_gene_pathways = {
        gene_id: gene_to_pathways.get(gene_id, set())
        for gene_id in kegg_gene_ids
    }

    n = len(kegg_gene_ids)
    matrix = np.zeros((n, n), dtype=float)

    for i, gene_a in enumerate(kegg_gene_ids):
        for j, gene_b in enumerate(kegg_gene_ids):
            if i == j:
                matrix[i, j] = 1.0
            else:
                matrix[i, j] = pathway_comembership_score(
                    selected_gene_pathways[gene_a],
                    selected_gene_pathways[gene_b],
                )

    similarity_matrix = pd.DataFrame(
        matrix,
        index=kegg_gene_ids,
        columns=kegg_gene_ids,
    )

    return similarity_matrix, selected_gene_pathways



In [2]:
kegg_genes = [
    "hsa:7157",  # TP53
    "hsa:672",   # BRCA1
    "hsa:675",   # BRCA2
    "hsa:1956",  # EGFR
    "hsa:5290",  # PIK3CA
]

kegg_similarity, gene_pathways = build_kegg_pathway_comembership_matrix_from_kegg_ids(
    kegg_gene_ids=kegg_genes,
    organism="hsa",
)

print(kegg_similarity)

          hsa:7157   hsa:672   hsa:675  hsa:1956  hsa:5290
hsa:7157  1.000000  0.074074  0.056604  0.275000  0.311475
hsa:672   0.074074  1.000000  0.333333  0.054545  0.035714
hsa:675   0.056604  0.333333  1.000000  0.056604  0.027027
hsa:1956  0.275000  0.054545  0.056604  1.000000  0.311475
hsa:5290  0.311475  0.035714  0.027027  0.311475  1.000000


## Métrica 4 - Sequence Similarity (BLAST score)
La similitud de secuencia mediante BLAST estima qué tan parecidas son dos secuencias biológicas, ya sea de ADN, ARN o proteína. A diferencia de las métricas basadas en redes o anotaciones funcionales, BLAST compara directamente la composición de las secuencias y busca regiones locales de alta similitud. Por ello, resulta útil para inferir homología, relaciones evolutivas, conservación de dominios o posible equivalencia funcional entre genes o proteínas.

BLAST no entrega una única métrica, sino varios indicadores complementarios. Los más relevantes son:

- **Porcentaje de identidad:** proporción de posiciones alineadas que son idénticas entre ambas secuencias.
- **Cobertura del alineamiento:** proporción de la secuencia consultada o de referencia cubierta por el alineamiento.
- **Bit score:** puntaje normalizado del alineamiento; valores mayores indican alineamientos más fuertes.
- **E-value:** número esperado de alineamientos similares que podrían aparecer por azar en una base de datos del mismo tamaño; valores menores indican mayor significancia estadística.

Una forma práctica de construir una similitud gen–gen es normalizar el bit score respecto al máximo posible para cada secuencia:

$$sim_{BLAST}(g_i,g_j)=\frac{bitscore(g_i,g_j)}{\max(bitscore(g_i,g_i),bitscore(g_j,g_j))}$$

Otra alternativa más restrictiva es considerar únicamente pares con alineamientos recíprocos fuertes, por ejemplo mediante _reciprocal best hits_, lo cual es especialmente útil para inferir ortología entre especies.

Para evitar asociaciones espurias, conviene exigir simultáneamente un E-value bajo, cobertura suficiente y porcentaje de identidad mínimo. Por ejemplo, en análisis exploratorios pueden filtrarse alineamientos con $Evalue < 10^{-5}$, cobertura mayor al 50 % e identidad mínima definida según el tipo de secuencia. Estos valores no son universales: secuencias cortas, familias génicas grandes o dominios conservados requieren criterios más cuidadosos.

Si se requiere una matriz de distancia, se puede transformar la similitud normalizada como:

$$d_{BLAST}(g_i,g_j)=1-sim_{BLAST}(g_i,g_j)$$

Esta métrica aporta una perspectiva estructural/evolutiva. Sin embargo, alta similitud de secuencia no siempre implica misma función, y baja similitud no descarta convergencia funcional o participación en una misma vía biológica.

# Referencias
1. Szklarczyk, D., Nastou, K., Koutrouli, M., Kirsch, R., Mehryary, F., Hachilif, R., Hu, D., Peluso, M. E., Huang, Q., Fang, T., Doncheva, N. T., Pyysalo, S., Bork, P., Jensen, L. J., & von Mering, C. (2025). The STRING database in 2025: protein networks with directionality of regulation. Nucleic acids research, 53(D1), D730–D737. https://doi.org/10.1093/nar/gkae1113
2. STRING help. (s. f.). https://string-db.org/cgi/help?sessionId=bxzyyZR4vKxc 
3. Piñero, J., Ramírez-Anguita, J. M., Saüch-Pitarch, J., Ronzano, F., Centeno, E., Sanz, F., & Furlong, L. I. (2019). The DisGeNET knowledge platform for disease genomics: 2019 update. Nucleic Acids Research, 48(D1), D845-D855. https://doi.org/10.1093/nar/gkz1021 
4. DISGENET. (s. f.). What is the DISGENET Score? Examples of GDA & VDA Score. DISGENET Help & Support. Recuperado 1 de mayo de 2026, de https://support.disgenet.com/support/solutions/articles/202000100283-what-are-the-gda-score-vda-score-disgenet-score- 
5. Parikh, J. R., Xia, Y., & Marto, J. A. (2012b). Multi-Edge Gene Set Networks Reveal Novel Insights into Global Relationships between Biological Themes. PLoS ONE, 7(9), e45211. https://doi.org/10.1371/journal.pone.0045211 
6. 